# dots.ocr Document Parser

vLLM 엔드포인트(dots.mocr)를 활용한 PDF/이미지 문서 OCR → Markdown 변환 노트북

**파이프라인:**  
`PDF/이미지` → `페이지 이미지 변환` → `vLLM OCR (병렬)` → `레이아웃 JSON` → `Markdown 조립` → `저장/미리보기`

**처음 실행 시:** Cell 1 (패키지 설치) 먼저 실행 → 이후부터는 Cell 2부터 바로 실행 가능

## Cell 1. 패키지 설치 (최초 1회만)

In [ ]:
# 이 노트북과 같은 폴더에 있는 wheels/ 디렉토리를 이용한 오프라인 설치
import subprocess, sys, pathlib

notebook_dir = pathlib.Path(".").resolve()
wheels_dir   = notebook_dir / "wheels"

assert wheels_dir.exists(), f"wheels/ 폴더가 없습니다: {wheels_dir}"

result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-index",
        "--find-links", str(wheels_dir),
        "PyMuPDF", "openai", "Pillow", "tqdm", "pydantic", "requests",
    ],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("[오류]", result.stderr[-1000:])

## Cell 2. Config — 여기만 수정하면 됩니다

In [ ]:
import sys, os, pathlib

# ── dots_ocr 패키지 경로 (같은 폴더에 있는 dots_ocr/)
NOTEBOOK_DIR = pathlib.Path(".").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

# ============================
#  vLLM 서버 설정
# ============================
VLLM_HOST     = "http://XXXXX"   # 예: "http://192.168.1.100:8000"
MODEL_NAME    = "model"           # vllm serve 시 --served-model-name 값
TEMPERATURE   = 0.1
TOP_P         = 1.0
MAX_TOKENS    = 16384

# ============================
#  입력 / 출력
# ============================
INPUT_PATH    = "./sample.pdf"    # PDF 또는 이미지 파일 경로
OUTPUT_DIR    = "./output"        # 결과 저장 폴더

# ============================
#  처리 옵션
# ============================
DPI           = 200               # PDF 렌더링 DPI (높을수록 정확, 느림)
NUM_THREADS   = 8                 # 병렬 페이지 처리 스레드 수
PROMPT_MODE   = "prompt_layout_all_en"  # 아래 주석 참고

# PROMPT_MODE 옵션:
#   "prompt_layout_all_en"  — 레이아웃 감지 + OCR → JSON + MD (기본, 권장)
#   "prompt_ocr"            — 텍스트 추출만 (빠름, 헤더/푸터 제외)
#   "prompt_layout_only_en" — 레이아웃 bbox 감지만 (텍스트 없음)

SHOW_BBOX_VIZ = True              # 레이아웃 bbox 시각화 표시 여부
SAVE_JSON     = True              # 페이지별 JSON 저장 여부

print(f"설정 완료")
print(f"  vLLM  : {VLLM_HOST}")
print(f"  model : {MODEL_NAME}")
print(f"  입력  : {INPUT_PATH}")
print(f"  출력  : {OUTPUT_DIR}")

## Cell 3. 핵심 함수 정의

In [ ]:
import json
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

from PIL import Image
from openai import OpenAI
from tqdm.notebook import tqdm
from IPython.display import display, Markdown

# ── dots_ocr 로컬 모듈
from dots_ocr.utils.prompts import dict_promptmode_to_prompt
from dots_ocr.utils.layout_utils import post_process_output, draw_layout_on_image
from dots_ocr.utils.format_transformer import layoutjson2md
from dots_ocr.utils.doc_utils import load_images_from_pdf
from dots_ocr.utils.image_utils import PILimage_to_base64, fetch_image


def call_vllm(image: Image.Image, prompt: str):
    """PIL 이미지 + 프롬프트 → vLLM 응답 텍스트"""
    client = OpenAI(
        api_key=os.environ.get("API_KEY", "0"),
        base_url=f"{VLLM_HOST}/v1",
    )
    try:
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {"url": PILimage_to_base64(image)},
                        },
                        # vLLM v1에서 이미지 토큰 위치 명시 필요
                        {"type": "text", "text": f"<|img|><|imgpad|><|endofimg|>{prompt}"},
                    ],
                }
            ],
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_completion_tokens=MAX_TOKENS,
        )
        return resp.choices[0].message.content
    except Exception as e:
        print(f"[vLLM 오류] {e}")
        return None


def process_page(page_idx: int, origin_image: Image.Image, prompt_mode: str) -> dict:
    """한 페이지 → OCR 결과 dict"""
    image = fetch_image(origin_image)   # RGB 변환
    prompt = dict_promptmode_to_prompt[prompt_mode]
    response = call_vllm(image, prompt)

    result = {"page_idx": page_idx, "origin_image": origin_image, "raw_response": response}

    if response is None:
        result.update({"cells": [], "md": "", "md_nohf": "", "error": "vLLM 응답 없음"})
        return result

    if prompt_mode in ("prompt_layout_all_en", "prompt_layout_only_en"):
        cells, filtered = post_process_output(response, prompt_mode, origin_image, image)
        result["cells"]    = cells if not filtered else []
        result["filtered"] = filtered

        if prompt_mode != "prompt_layout_only_en" and not filtered:
            result["md"]      = layoutjson2md(origin_image, cells, text_key="text")
            result["md_nohf"] = layoutjson2md(origin_image, cells, text_key="text", no_page_hf=True)
        else:
            result["md"]      = cells if filtered else ""
            result["md_nohf"] = result["md"]
    else:
        # prompt_ocr 등 — 응답이 곧 텍스트
        result.update({"cells": [], "md": response, "md_nohf": response})

    return result


print("함수 정의 완료")

## Cell 4. PDF / 이미지 → 페이지 이미지 변환

In [ ]:
input_path = pathlib.Path(INPUT_PATH)
assert input_path.exists(), f"파일 없음: {input_path}"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

if input_path.suffix.lower() == ".pdf":
    print(f"PDF 로드 중: {input_path}  (DPI={DPI})")
    page_images = load_images_from_pdf(str(input_path), dpi=DPI)
    print(f"총 {len(page_images)} 페이지")
elif input_path.suffix.lower() in IMAGE_EXTS:
    page_images = [Image.open(str(input_path)).convert("RGB")]
    print(f"이미지 1장 로드: {input_path}")
else:
    raise ValueError(f"지원하지 않는 형식: {input_path.suffix}")

# 첫 페이지 미리보기
preview = page_images[0].copy()
preview.thumbnail((600, 800))
display(preview)

## Cell 5. OCR — vLLM 병렬 처리

In [ ]:
total_pages  = len(page_images)
page_results = [None] * total_pages

print(f"{total_pages} 페이지를 {min(NUM_THREADS, total_pages)} 스레드로 처리합니다...")
print(f"Prompt mode: {PROMPT_MODE}")

with ThreadPoolExecutor(max_workers=min(NUM_THREADS, total_pages)) as executor:
    future_to_idx = {
        executor.submit(process_page, i, img, PROMPT_MODE): i
        for i, img in enumerate(page_images)
    }
    with tqdm(total=total_pages, desc="OCR 진행") as pbar:
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                page_results[idx] = future.result()
            except Exception as e:
                print(f"[오류] 페이지 {idx+1}: {e}")
                page_results[idx] = {
                    "page_idx": idx, "origin_image": page_images[idx],
                    "cells": [], "md": "", "md_nohf": "", "error": str(e),
                }
            pbar.update(1)

success  = sum(1 for r in page_results if r and not r.get("error"))
filtered = sum(1 for r in page_results if r and r.get("filtered"))
print(f"\n완료: {success}/{total_pages} 성공  |  {filtered}개 JSON 파싱 fallback")

## Cell 6. Markdown 조립

In [ ]:
stem = input_path.stem
parts, parts_nohf = [], []

for r in page_results:
    page_no = r["page_idx"] + 1
    parts.append(f"\n\n<!-- Page {page_no} -->\n")
    parts.append(r.get("md") or "")
    parts_nohf.append(f"\n\n<!-- Page {page_no} -->\n")
    parts_nohf.append(r.get("md_nohf") or r.get("md") or "")

final_md      = "\n".join(parts).strip()
final_md_nohf = "\n".join(parts_nohf).strip()

print(f"Markdown 길이: {len(final_md):,} 자")
print("\n--- 앞부분 미리보기 ---")
print(final_md[:600])

## Cell 7. 결과 저장

In [ ]:
out_dir = pathlib.Path(OUTPUT_DIR) / stem
out_dir.mkdir(parents=True, exist_ok=True)

# ── Markdown 저장
md_path      = out_dir / f"{stem}.md"
md_nohf_path = out_dir / f"{stem}_nohf.md"
md_path.write_text(final_md, encoding="utf-8")
md_nohf_path.write_text(final_md_nohf, encoding="utf-8")
print(f"저장: {md_path}")
print(f"저장: {md_nohf_path}")

# ── 페이지별 JSON 저장 (옵션)
if SAVE_JSON:
    json_dir = out_dir / "json"
    json_dir.mkdir(exist_ok=True)
    for r in page_results:
        cells = r.get("cells", [])
        if cells:
            p = json_dir / f"{stem}_page_{r['page_idx']+1:03d}.json"
            p.write_text(json.dumps(cells, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"JSON 저장: {json_dir}")

# ── 레이아웃 시각화 이미지 저장 (옵션)
if SHOW_BBOX_VIZ and PROMPT_MODE == "prompt_layout_all_en":
    viz_dir = out_dir / "layout_viz"
    viz_dir.mkdir(exist_ok=True)
    for r in page_results:
        cells = r.get("cells", [])
        if cells and not r.get("filtered"):
            try:
                viz = draw_layout_on_image(r["origin_image"], cells)
                viz.save(str(viz_dir / f"{stem}_page_{r['page_idx']+1:03d}_layout.jpg"))
            except Exception as e:
                print(f"[시각화 오류] 페이지 {r['page_idx']+1}: {e}")
    print(f"레이아웃 시각화 저장: {viz_dir}")

print("\n모든 결과 저장 완료")

## Cell 8. Jupyter 인라인 미리보기

In [ ]:
PREVIEW_PAGES = 3  # 미리볼 페이지 수

for r in page_results[:PREVIEW_PAGES]:
    page_no = r["page_idx"] + 1
    md_text = r.get("md_nohf") or r.get("md") or "*(OCR 결과 없음)*"
    # base64 이미지는 제거 (미리보기 속도)
    md_text = re.sub(r"!\[\]\(data:image[^)]+\)", "*(이미지 생략)*", md_text)
    display(Markdown(f"---\n## Page {page_no}\n{md_text}"))

In [ ]:
# 레이아웃 bbox 시각화 미리보기
if SHOW_BBOX_VIZ and PROMPT_MODE == "prompt_layout_all_en":
    PREVIEW_VIZ = 2
    for r in page_results[:PREVIEW_VIZ]:
        cells = r.get("cells", [])
        if not cells or r.get("filtered"):
            print(f"Page {r['page_idx']+1}: bbox 시각화 불가")
            continue
        try:
            viz = draw_layout_on_image(r["origin_image"], cells)
            viz.thumbnail((800, 1100))
            print(f"--- Page {r['page_idx']+1} 레이아웃 ---")
            display(viz)
        except Exception as e:
            print(f"Page {r['page_idx']+1} 시각화 오류: {e}")

## Cell 9. 통계 요약

In [ ]:
from collections import Counter

cat_counter = Counter()
error_pages = []

for r in page_results:
    if r.get("error"):
        error_pages.append(r["page_idx"] + 1)
    for cell in r.get("cells", []):
        cat_counter[cell.get("category", "Unknown")] += 1

print(f"{'파일':<20} {input_path.name}")
print(f"{'총 페이지':<20} {total_pages}")
print(f"{'성공':<20} {total_pages - len(error_pages)}")
print(f"{'실패 페이지':<20} {error_pages or '없음'}")
print(f"{'Markdown 크기':<20} {len(final_md):,} 자")

if cat_counter:
    print("\n레이아웃 카테고리 분포:")
    for cat, cnt in sorted(cat_counter.items(), key=lambda x: -x[1]):
        bar = "█" * min(cnt, 40)
        print(f"  {cat:<22} {cnt:>4}  {bar}")